# 02 — Tools & Agents (`create_agent`)

This is the notebook where a lot of stale tutorials will mislead you. **Deprecated (do not use):**
- `from langchain.agents import initialize_agent`
- `from langchain.agents import AgentExecutor`
- `AgentType.ZERO_SHOT_REACT_DESCRIPTION`

**Current (v1.0+):** `from langchain.agents import create_agent`. It builds a LangGraph-backed agent under the hood, but you interact with it through a simple high-level API.


In [ ]:
import os
from getpass import getpass
from langchain.chat_models import init_chat_model

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
MODEL_ID = "openai:gpt-4.1-mini"

model = init_chat_model(MODEL_ID, temperature=0)

## 1. Defining a tool

The `@tool` decorator turns a normal Python function into something the model can call. **The docstring matters** — the model reads it to decide when and how to use the tool, so write it like documentation, not a comment.


In [ ]:
from langchain_core.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the latest price for a stock ticker symbol (e.g. 'TCS', 'INFY', 'RELIANCE').

    Args:
        ticker: The stock ticker symbol, uppercase, no exchange suffix.
    """
    # Mocked for the notebook — replace with a real API call (e.g. Alpha Vantage, NSE API)
    fake_prices = {"TCS": 4123.50, "INFY": 1567.20, "RELIANCE": 2984.10}
    price = fake_prices.get(ticker.upper())
    if price is None:
        return f"No data found for {ticker}"
    return f"{ticker.upper()} is currently trading at ₹{price}"


@tool
def calculate_percentage_change(old_value: float, new_value: float) -> str:
    """Calculate the percentage change between an old and a new value.

    Args:
        old_value: The starting value.
        new_value: The ending value.
    """
    change = ((new_value - old_value) / old_value) * 100
    return f"{change:.2f}%"

tools = [get_stock_price, calculate_percentage_change]

## 2. Building the agent with `create_agent`

This is the current standard pattern. `model` can be a string ID or an already-initialized chat model object.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful financial assistant for Indian equities. "
        "Use the tools available to answer questions about stock prices and percentage changes. "
        "Always show your reasoning briefly before the final answer."
    ),
)

## 3. Running the agent

`create_agent` returns a compiled LangGraph graph, so you call it with `.invoke()` passing a `messages` list — same shape LangGraph uses everywhere.


In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "What's the price of TCS, and how much would I gain in percentage terms if I bought at 4000 and it's now at that price?"}]
})

for m in result["messages"]:
    role = m.__class__.__name__
    print(f"[{role}] {m.content}")

### Reading the trace
Notice the message list includes the tool calls and tool results, not just the final answer — this is the full "trace" of the agent's reasoning. In production you'd log this for debugging.


## 4. Streaming an agent's steps

For longer agent runs, streaming lets you show progress instead of a long silent wait.


In [ ]:
for step in agent.stream(
    {"messages": [{"role": "user", "content": "What's INFY trading at right now?"}]},
    stream_mode="values",
):
    last_message = step["messages"][-1]
    last_message.pretty_print()

## 5. A tool that hits a real external source (web search example, optional)

If you want your agent to answer questions beyond its training data, wire in a real tool. Below is the shape you'd use with `langchain-tavily` (a search provider integration) — commented out since it needs its own API key. This shows you *where* real-world tools slot in.


In [ ]:
# %pip install -q langchain-tavily
# import os
# os.environ["TAVILY_API_KEY"] = getpass("Enter your TAVILY_API_KEY: ")
# from langchain_tavily import TavilySearch
#
# search_tool = TavilySearch(max_results=3)
# research_agent = create_agent(model=model, tools=[search_tool], system_prompt="You research questions using web search.")
# out = research_agent.invoke({"messages": [{"role": "user", "content": "What is LangChain's latest major version?"}]})
# out["messages"][-1].pretty_print()

---
### Key takeaways
- `@tool` + a good docstring is how the model learns what a function does and when to call it.
- `create_agent(model=..., tools=..., system_prompt=...)` is the current, standard agent constructor.
- The agent is invoked with `{"messages": [...]}` and returns the full message trace, including tool calls.
- `initialize_agent` / `AgentExecutor` are deprecated — avoid them in new code.

**Next:** `03_memory_and_state.ipynb` — giving the agent memory across turns and sessions.
